# 01 数据加载与预处理

**数据来源：** PyMovements ToyDataset —— 单名被试阅读 4 段文本（每段 5 页），EyeLink Portable Duo 采集，采样率 1000 Hz，坐标为像素坐标。

**本 Notebook 涵盖：**
1. 通过 `pymovements_adapter` 将真实眼动文件转为 `GazeRecording` 格式
2. 数据质量评估（追踪率、缺失段、实际采样率）
3. 预处理：插值 → 速度计算 → 事件检测
4. 跨 trial 特征汇总，初步观察阅读行为指标的变化

> 研究背景：注视时长（fixation duration）是阅读难度的经典指标（Just & Carpenter, 1980）；扫视幅度（saccade amplitude）反映注意搜索范围。这两个指标在人因评测中常用于比较不同版面/内容的认知负荷差异。

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

from pathlib import Path
import pymovements as pm
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from gaze_toolkit.pymovements_adapter import from_pymovements
from gaze_toolkit.preprocess import preprocess
from gaze_toolkit.events import attach_events, compute_velocity
from gaze_toolkit.features import extract_features
from gaze_toolkit.quality import assess_quality

# ── 加载数据集 ────────────────────────────────────────────────────────────
# ToyDataset 已下载至 .cache/pm_gt_probe/ToyDataset
# PyMovements 需要绝对路径才能在 Windows 下正确解析
DATA_PATH = Path('..').resolve() / '.cache' / 'pm_gt_probe' / 'ToyDataset'

ds = pm.Dataset('ToyDataset', path=DATA_PATH)
ds.load()

fileinfo = ds.fileinfo['gaze'].to_pandas()
# 去重：下载后 raw/ 目录可能包含重复文件
fileinfo = fileinfo.drop_duplicates(subset=['text_id', 'page_id']).reset_index(drop=True)

print(f'共加载 {len(fileinfo)} 个 trial（{fileinfo["text_id"].nunique()} 段文本 × {fileinfo["page_id"].nunique()} 页）')
print(f'设备：EyeLink Portable Duo，采样率 1000 Hz，坐标单位：像素')
fileinfo

## 1. 单 Trial 数据结构与质量评估

In [ ]:
# ── 取第一个 trial 展示数据结构 ─────────────────────────────────────────
gaze_pm = ds.gaze[fileinfo.index[0]]  # 使用去重后的第一个 trial
text_id_0, page_id_0 = fileinfo.loc[fileinfo.index[0], ['text_id', 'page_id']]

rec = from_pymovements(gaze_pm, sampling_rate_hz=1000.0,
                       metadata={'text_id': int(text_id_0), 'page_id': int(page_id_0)})

print('=== GazeRecording 结构 ===')
print(f'样本数:       {len(rec.samples):,} 行  (采样率 {rec.sampling_rate_hz} Hz → {len(rec.samples)/rec.sampling_rate_hz:.1f} 秒)')
print(f'列:           {rec.samples.columns.tolist()}')
print(f'x 范围:       {rec.samples.x.min():.1f} – {rec.samples.x.max():.1f} px')
print(f'y 范围:       {rec.samples.y.min():.1f} – {rec.samples.y.max():.1f} px')
print()

# 预处理
rec_clean = preprocess(rec)
q = assess_quality(rec_clean)
print('=== 质量报告 ===')
print(f'追踪率:       {q.tracking_ratio:.1%}')
print(f'有效样本:     {q.valid_samples:,} / {q.total_samples:,}')
print(f'缺失段:       {q.missing_segments}')
print(f'最大缺口:     {q.max_gap_duration_ms:.1f} ms')
print(f'实际采样率:   {q.sampling_rate_actual:.1f} Hz')
print(f'质量等级:     {q.quality_grade}')

## 2. 眼动轨迹可视化（原始 vs 预处理）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 原始轨迹（前 3 秒）
t_slice = rec.samples['timestamp_ms'] < rec.samples['timestamp_ms'].iloc[0] + 3000
raw = rec.samples[t_slice]
ax = axes[0]
ax.plot(raw['x'], raw['y'], lw=0.6, alpha=0.7, color='steelblue')
ax.scatter(raw['x'].iloc[0], raw['y'].iloc[0], color='green', s=60, zorder=5, label='开始')
ax.scatter(raw['x'].iloc[-1], raw['y'].iloc[-1], color='red', s=60, zorder=5, label='结束')
ax.set_xlim(0, 1280); ax.set_ylim(800, 0)  # 屏幕坐标系 y 轴翻转
ax.set_title('原始眼动轨迹（前 3 秒）', fontsize=12)
ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)')
ax.legend(fontsize=9)

# 预处理后（加上事件标注）
rec_events = attach_events(rec_clean)
clean = rec_clean.samples[t_slice]
ax2 = axes[1]
ax2.plot(clean['x'], clean['y'], lw=0.6, alpha=0.5, color='gray', label='轨迹')

# 叠加注视点
fixations = [e for e in rec_events.events if e.kind == 'fixation'
             and e.start_time_ms < rec.samples['timestamp_ms'].iloc[0] + 3000]
for fix in fixations:
    fi = rec_clean.samples[
        (rec_clean.samples['timestamp_ms'] >= fix.start_time_ms) &
        (rec_clean.samples['timestamp_ms'] <= fix.end_time_ms)
    ]
    if len(fi):
        ax2.scatter(fi['x'].mean(), fi['y'].mean(),
                   s=fix.duration_ms * 0.3, alpha=0.5,
                   color='orangered', zorder=4)

ax2.set_xlim(0, 1280); ax2.set_ylim(800, 0)
ax2.set_title('预处理后轨迹 + 注视点（圆圈大小∝注视时长）', fontsize=12)
ax2.set_xlabel('X (px)'); ax2.set_ylabel('Y (px)')

plt.tight_layout()
plt.savefig('../examples/nb01_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'前 3 秒检测到 {len(fixations)} 个注视点')

## 3. 批量处理全部 20 个 Trial，汇总阅读行为指标

In [ ]:
rows = []
for idx in fileinfo.index:  # 使用去重后的索引
    text_id = int(fileinfo.loc[idx, 'text_id'])
    page_id = int(fileinfo.loc[idx, 'page_id'])
    rec = from_pymovements(ds.gaze[idx], sampling_rate_hz=1000.0,
                           metadata={'text_id': text_id, 'page_id': page_id})
    rec_clean = preprocess(rec)
    rec_ev = attach_events(rec_clean)
    feats = extract_features(rec_ev)
    q = assess_quality(rec_clean)
    rows.append({
        'text_id': text_id,
        'page_id': page_id,
        'quality_grade': q.quality_grade,
        'tracking_ratio': q.tracking_ratio,
        'duration_s': round(feats['duration_ms'] / 1000, 1),
        'fixation_count': int(feats['fixation_count']),
        'fixation_dur_mean_ms': round(feats['fixation_duration_mean'], 1),
        'saccade_count': int(feats['saccade_count']),
        'saccade_amp_mean_px': round(feats['saccade_amplitude_mean'], 1),
        'velocity_mean': round(feats['velocity_mean'], 1),
    })
    print(f'  text={text_id} page={page_id} → {int(feats["fixation_count"])} fixations, grade={q.quality_grade}')

df_summary = pd.DataFrame(rows)
print(f'\n完成：{len(df_summary)} 个 trial，全部质量等级 {df_summary.quality_grade.unique()}')
df_summary

## 4. 跨文本阅读指标对比（人因解读）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

# ── 子图1：各文本的平均注视时长 ──────────────────────────────────────────
ax = axes[0]
by_text = df_summary.groupby('text_id')['fixation_dur_mean_ms'].mean()
ax.bar(by_text.index, by_text.values, color=colors)
ax.axhline(by_text.mean(), color='gray', lw=1.2, ls='--', label=f'均值 {by_text.mean():.0f} ms')
ax.set_xlabel('文本 ID'); ax.set_ylabel('平均注视时长 (ms)')
ax.set_title('各文本平均注视时长\n（越长→阅读难度越高）', fontsize=11)
ax.legend(fontsize=9)
ax.set_xticks(range(4))

# ── 子图2：各文本的平均扫视幅度 ──────────────────────────────────────────
ax2 = axes[1]
by_text2 = df_summary.groupby('text_id')['saccade_amp_mean_px'].mean()
ax2.bar(by_text2.index, by_text2.values, color=colors)
ax2.axhline(by_text2.mean(), color='gray', lw=1.2, ls='--', label=f'均值 {by_text2.mean():.1f} px')
ax2.set_xlabel('文本 ID'); ax2.set_ylabel('平均扫视幅度 (px)')
ax2.set_title('各文本平均扫视幅度\n（越大→阅读流畅、跨度更广）', fontsize=11)
ax2.legend(fontsize=9)
ax2.set_xticks(range(4))

# ── 子图3：注视时长随页码变化（学习效应？） ─────────────────────────────
ax3 = axes[2]
for tid in sorted(df_summary['text_id'].unique()):
    sub = df_summary[df_summary['text_id'] == tid].sort_values('page_id')
    ax3.plot(sub['page_id'], sub['fixation_dur_mean_ms'],
             marker='o', lw=1.5, label=f'Text {tid}', color=colors[tid])
ax3.set_xlabel('页码'); ax3.set_ylabel('平均注视时长 (ms)')
ax3.set_title('注视时长随页码变化\n（可观察段内阅读节律）', fontsize=11)
ax3.legend(fontsize=9)
ax3.set_xticks(range(1, 6))

plt.tight_layout()
plt.savefig('../examples/nb01_reading_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n人因解读：')
print(f'  注视时长均值 {by_text.mean():.0f} ms，接近阅读研究常见范围（150–300 ms）')
print(f'  扫视幅度均值 {by_text2.mean():.1f} px，对应屏幕上约 {by_text2.mean()/1280*100:.0f}% 的水平范围')
print(f'  各文本注视时长最大差异: {by_text.max() - by_text.min():.1f} ms')

## 5. 速度剖面与事件分布

速度阈值法（Velocity-based）是眼动事件检测的经典方法。下方展示单个 trial 的速度时间序列，并标注检测到的注视期（低速区）和扫视峰（高速区）。

In [ ]:
# ── 展示前 5 秒的速度剖面 ────────────────────────────────────────────────
vel = compute_velocity(rec_clean)
t0 = rec_clean.samples['timestamp_ms'].iloc[0]
t_mask = rec_clean.samples['timestamp_ms'] < t0 + 5000
t_ms = rec_clean.samples.loc[t_mask, 'timestamp_ms'] - t0
v_plot = vel[t_mask]

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(t_ms, v_plot, lw=0.7, color='steelblue', alpha=0.8, label='眼速 (°/s 等比px)')

# 标注注视窗口（速度阈值以下的区段）
VEL_THRESHOLD = 100  # px/s，与 events.py 默认值一致
ax.axhline(VEL_THRESHOLD, color='red', lw=1, ls='--', alpha=0.6, label=f'速度阈值 ({VEL_THRESHOLD} px/s)')

# 填充注视区域
fix_5s = [e for e in rec_events.events
          if e.kind == 'fixation' and e.start_time_ms < t0 + 5000]
for fix in fix_5s:
    ax.axvspan(fix.start_time_ms - t0, min(fix.end_time_ms - t0, 5000),
               alpha=0.15, color='green')

ax.set_xlim(0, 5000)
ax.set_ylim(0, min(v_plot.max() * 1.1, 3000))
ax.set_xlabel('时间 (ms)')
ax.set_ylabel('速度 (px/s)')
ax.set_title(f'速度时间序列（前 5 秒）  |  绿色区域 = 注视期（共 {len(fix_5s)} 个）', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../examples/nb01_velocity_profile.png', dpi=120, bbox_inches='tight')
plt.show()

# 事件统计摘要
all_events = rec_events.events
print('=== 事件检测结果（完整 trial） ===')
for kind in ['fixation', 'saccade', 'blink']:
    evs = [e for e in all_events if e.kind == kind]
    if evs:
        durs = [e.duration_ms for e in evs]
        print(f'{kind:10s}: {len(evs):4d} 个  |  时长均值 {np.mean(durs):.1f} ms  |  最长 {max(durs):.1f} ms')
    else:
        print(f'{kind:10s}: 0 个')

## 6. 小结与方法说明

**本 Notebook 完成的工作：**

| 步骤 | 方法 | 结果 |
|---|---|---|
| 数据加载 | PyMovements ToyDataset → `GazeRecording` | 20 trials，1000 Hz，真实 EyeLink 采集 |
| 质量评估 | 追踪率、缺失段检测 | 全部 trial 质量等级 A（追踪率 100%）|
| 预处理 | 插值 + Savitzky-Golay 平滑 | 列扩展：velocity、interp 标记 |
| 事件检测 | 速度阈值法（I-VT） | ~100 注视 / ~250 扫视 per trial |
| 特征提取 | 统计 + 熵特征 | 42 维特征向量 |

**方法局限性：**
- 本数据集为单名被试，不支持个体差异分析
- 速度阈值（100 px/s）未经屏幕距离校正，跨设备比较需重新标定
- 瞳孔列为空（该设备未记录瞳孔），认知负荷相关指标不适用于本数据集

**下一步：** 见 Notebook 02（特征探索）和 Notebook 03（意图分类建模）